In [1]:
import sys
import os
import pandas as pd
import numpy as np
import yaml
from DaySim import DaysimSummary
from Survey import DaysimSummary_Survey
from calibration_utils import (load_beta_lookup, save_log, write_f12, run_1d, run_2d, _make_csv_key_fn, _get_raw)

c:\Users\USVA682771\OneDrive - WSP O365\Documents\Daysim_summaries\calibration_utils.py:148: SyntaxWarning: invalid decimal literal
  'variable_mean': 1.0if pd.isna(vm) else float(vm)}


## Run once per session

In [2]:
# Load model and survey objects
model  = DaysimSummary()
survey = DaysimSummary_Survey()

runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...
runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...


## Run for each calibration run
      - select model to run in calibration_config.yaml and run cells below
      - initially, review LOG_FILE and updated coefficient file in coefficient_files_interim_dir after each run. If works as expected, manually copy to the coefficient_files_input_dir to prepare for the next run
      - option to enable COPY_BACK to overwrite input coefficient file with updated coefficient file at the end of each run instead of manually copying

In [3]:
# Load config
with open('calibration_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

DAMPING_FACTOR = cfg['damping_factor']
THRESHOLD = cfg['threshold']   
COPY_BACK = cfg['copy_back']

TARGETS_FILE = cfg['paths']['targets_file']
MAPPING_FILE = cfg['paths']['mapping_file']  
LOG_FILE = cfg['paths']['log_file']

coefficient_files_input_dir = cfg['paths']['coefficient_files_input_dir']
coefficient_files_interim_dir = cfg['paths']['coefficient_files_interim_dir']

MODELS_TO_RUN = [m for m, enabled in cfg['models'].items() if enabled]

print(f'Models running calibration: {MODELS_TO_RUN}')

# Load targets and mapping 
targets_df = pd.read_csv(TARGETS_FILE)
targets_df["method_arg"] = targets_df["method_arg"].fillna('')
targets_df['key_mode'] = targets_df['key_mode'].fillna('')

mapping_df = pd.read_csv(MAPPING_FILE)
mapping_df["group"] = mapping_df["group"].fillna('').astype(str)
mapping_df["alternative"] = mapping_df["alternative"].astype(str)
mapping_df["end"] = pd.to_numeric(mapping_df["end"], errors='coerce')


Models running calibration: ['WorkTourMode']


In [4]:
print(mapping_df[mapping_df['f12_model'] == "WorkTourMode"][['group','alternative']])

    group     alternative
123           Drive Alone
124         Shared Ride 2
125        Shared Ride 3+
126          Walk-Transit
127         Drive-Transit
128                  Bike
129                  Walk


In [5]:
MODEL = 'PersonExactNumberOfTours'


In [6]:
model_targets = targets_df[targets_df['model'] == MODEL]

In [7]:
model_targets


,model,f12_model,label,type,method,method_arg,key_mode
60,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_work,2d,summary_day_pattern_tour_count,work,count_by_purpose
61,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_school,2d,summary_day_pattern_tour_count,school,count_by_purpose
62,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_escort,2d,summary_day_pattern_tour_count,escort,count_by_purpose
63,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_pb,2d,summary_day_pattern_tour_count,pb,count_by_purpose
64,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_shop,2d,summary_day_pattern_tour_count,shop,count_by_purpose
65,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_meal,2d,summary_day_pattern_tour_count,meal,count_by_purpose
66,PersonExactNumberOfTours,PersonExactNumberOfTours,DayPatternCount_socrec,2d,summary_day_pattern_tour_count,socrec,count_by_purpose


In [8]:
mapping_df[mapping_df['f12_model'] == MODEL][['f12_model','group','alternative','end']]

,f12_model,group,alternative,end
100,PersonExactNumberOfTours,work,2,152.0
101,PersonExactNumberOfTours,work,3,153.0
102,PersonExactNumberOfTours,school,2,252.0
103,PersonExactNumberOfTours,school,3,253.0
104,PersonExactNumberOfTours,escort,2,352.0
105,PersonExactNumberOfTours,escort,3,353.0
106,PersonExactNumberOfTours,pb,2,452.0
107,PersonExactNumberOfTours,pb,3,453.0
108,PersonExactNumberOfTours,shop,2,552.0
109,PersonExactNumberOfTours,shop,3,553.0


In [9]:

# Coefficient files
F12_FILES = {
    file.split('Coefficients_Chattanooga')[0]: file 
    for file in os.listdir(coefficient_files_input_dir) 
    if file.endswith('.F12')
}

# Run calibration for each enabled model
for MODEL in MODELS_TO_RUN:
    model_targets = targets_df[targets_df['model'] == MODEL]
    if model_targets.empty:
        print(f'No targets found for model {MODEL}. Exiting.')
        exit()

    f12_models_needed = model_targets['f12_model'].unique().tolist()
    mapping_sub = mapping_df[mapping_df['f12_model'].isin(f12_models_needed)]
    beta_lookup = load_beta_lookup(mapping_sub, F12_FILES, coefficient_files_input_dir)
    print(list(beta_lookup.keys())[:5])  # Print first 5 keys to verify loading
# Run calibration
    log = []
    for _, cfg in model_targets.iterrows():
        try:
            m_raw = _get_raw(model, cfg['method'], cfg['method_arg'])
            s_raw = _get_raw(survey, cfg['method'], cfg['method_arg'])
            csv_key_fn = _make_csv_key_fn(cfg['key_mode'], cfg['method_arg'])
            runner = run_2d if cfg['type'] == '2d' else run_1d
            log += runner(cfg['label'], cfg['f12_model'], m_raw, s_raw, 
                               beta_lookup, DAMPING_FACTOR, THRESHOLD)
        except Exception as e:
            print(f'Error processing target {cfg["label"]}: {e}')

    comp_df = pd.DataFrame(log)
    calibration_rows= comp_df[comp_df['calibrate']]
    n_within = calibration_rows['within_threshold'].sum()
    n_total = len(calibration_rows)
    print(f'Calibration complete: {n_within} of {n_total} targets within threshold after adjustment.')


# Save log
save_log(comp_df, LOG_FILE)

# Write updated F12 files
to_update = comp_df[comp_df['calibrate'] & 
                    ~comp_df['within_threshold'] & 
                    (comp_df['adjustment']!= 0) &
                    comp_df['end'].notna() &
                    comp_df['new_beta'].notna()]

if to_update.empty:
    print('No updates needed for F12 files based on calibration results.')
else:
    print(f'Updating F12 files for {to_update["f12model"].nunique()} models based on calibration results...')
    for f12_model,grp in to_update.groupby('f12model'):
        if f12_model not in F12_FILES:
            print(f'Warning: No F12 file found for model {f12_model}, skipping update.')
            continue
        end_to_beta = {int(r['end']): round(r['new_beta'], 12) 
                       for _, r in grp.iterrows()}
        write_f12(f12_model, 
                  end_to_beta, 
                  f12_files = F12_FILES, 
                  coefficient_input_dir = coefficient_files_input_dir,
                  coefficient_interim_dir = coefficient_files_interim_dir,
                  copy_back = COPY_BACK)


[('WorkTourMode', '', 'Drive Alone'), ('WorkTourMode', '', 'Shared Ride 2'), ('WorkTourMode', '', 'Shared Ride 3+'), ('WorkTourMode', '', 'Walk-Transit'), ('WorkTourMode', '', 'Drive-Transit')]
('run_id', 'Drive Alone', '', 'Drive Alone')
('WorkTourMode', '', 'Drive Alone')
('run_id', 'Shared Ride 2', '', 'Shared Ride 2')
('WorkTourMode', '', 'Shared Ride 2')
('run_id', 'Shared Ride 3+', '', 'Shared Ride 3+')
('WorkTourMode', '', 'Shared Ride 3+')
('run_id', 'Walk-Transit', '', 'Walk-Transit')
('WorkTourMode', '', 'Walk-Transit')
('run_id', 'Drive-Transit', '', 'Drive-Transit')
('WorkTourMode', '', 'Drive-Transit')
('run_id', 'Bike', '', 'Bike')
('WorkTourMode', '', 'Bike')
Calibration complete: 6 of 6 targets within threshold after adjustment.
Saved calibration_log.csv (run9)
No updates needed for F12 files based on calibration results.


In [10]:

F12_FILES = {
    file.split('Coefficients_Chattanooga')[0]: file 
    for file in os.listdir(coefficient_files_input_dir) 
    if file.endswith('.F12')
}

from calibration_utils import read_f12
path = os.path.join(coefficient_files_input_dir, F12_FILES["WorkTourMode"])
with open(path, 'r') as f:
    for i, line in enumerate(f):
        if i < 10:  # Print first 10 lines for inspection
            print(line.strip())


Work tour mode choice
Created by ALOGIT version 4                             14:02:09 on  2 Apr 12
END
1  costutil  F  0.528792256   .191310023044
2  timeutil  F  0.558792764   .957386348522E-01
10  dt-const  F  -2.318982633  .742734994859
11 dt-nocars  T  -999            .000000000000
13 dt-carsltw F  -1.010720718  1.06878667775
20  wt-const  F  -3.446832801  1.11172541893
30  s3-const  F  -1.007601567 .722338259772


In [11]:
path

'9_Coefficients\\WorkTourModeCoefficients_Chattanooga.F12'

In [12]:
s = read_f12(path)
s.head()

""


In [14]:
beta_lookup

{('WorkTourMode', '', 'Drive Alone'): {'end': 50,
  'beta': nan,
  'variable_mean': 1.0},
 ('WorkTourMode', '', 'Shared Ride 2'): {'end': 40,
  'beta': nan,
  'variable_mean': 1.0},
 ('WorkTourMode', '', 'Shared Ride 3+'): {'end': 30,
  'beta': nan,
  'variable_mean': 1.0},
 ('WorkTourMode', '', 'Walk-Transit'): {'end': 20,
  'beta': nan,
  'variable_mean': 1.0},
 ('WorkTourMode', '', 'Drive-Transit'): {'end': 10,
  'beta': nan,
  'variable_mean': 1.0},
 ('WorkTourMode', '', 'Bike'): {'end': 60, 'beta': nan, 'variable_mean': 1.0},
 ('WorkTourMode', '', 'Walk'): {'end': 70, 'beta': nan, 'variable_mean': 1.0}}